################ CELL00: NOTEBOOK OVERVIEW ################
# WILLMA Whisper Transcriber V05f

This notebook provides a local-PC batch transcription workflow for audio files stored in a shared Research Drive location.

It is organized as separate sections for:

- setup and notebook usage
- dependency handling
- configuration
- Research Drive shared-folder discovery
- audio discovery and validation
- Whisper model loading
- single-file and batch transcription
- transcript formatting and saving
- progress and error reporting
- end-to-end execution
- preview and verification

The workflow is designed for use in Visual Studio Code with Jupyter support.

**V05f focus:** input files are no longer taken from `DEFAULT_INPUT_FOLDER`; instead, the notebook reads audio files from the Research Drive share defined by `SHARE_LINK` and `SHARE_PASSWORD` in `.whisper-env`.

################ CELL00A: V05f IMPROVEMENTS ################
# What's new in V05f

V05f builds on V05e and replaces local input-folder discovery with a split Research Drive workflow.

### Changes in V05f

#### 1 — Input comes from the public share, output goes to writable `RD:` storage
The notebook now expects these additional keys in `.whisper-env`:

| Key | Required | Purpose |
|---|---|---|
| `SHARE_LINK` | Yes | Password-protected Research Drive share URL used as the notebook input source |
| `SHARE_PASSWORD` | Yes | Password for the protected Research Drive share |
| `RD_OUTPUT_PATH` | Yes | Writable `RD:` destination where all generated output files and folders are copied |

`DEFAULT_INPUT_FOLDER` is no longer used as the transcription input source in V05f.

#### 2 — Public-share read path for input
The notebook reads source audio from the public Research Drive share by:
- extracting the share token from `SHARE_LINK`
- converting `SHARE_PASSWORD` with `rclone obscure`
- building the Nextcloud DAV-compatible `rclone` source
- copying the shared audio files into a local cache before transcription starts

This keeps input access independent from any personal `RD:` mount or browser URL.

#### 3 — Writable `RD:` path for output delivery
The notebook writes outputs in two stages:
- first, it generates all transcript files locally in the notebook output folder
- second, it copies those local results to `RD_OUTPUT_PATH`

`RD_OUTPUT_PATH` must be a normal writable `rclone` destination such as:

```text
RD:HR-DATALAB-HEALTHCARE (Projectfolder)/TRANSCRIBED_WILLMA_OUTPUTS/
```

The public `SHARE_LINK` itself is read-only and is not used for saving outputs.

#### 4 — Better inline comments
The share-access and output-sync logic is explicitly commented so it is clear which part:
- parses the public share URL
- builds the DAV-compatible `rclone` source for input
- validates the writable `RD:` destination for output
- copies local results to Research Drive after processing

#### 5 — Existing transcription pipeline retained
Once files are available locally from the shared source, the rest of the workflow remains the same: validation, transcription, diarization, output verification, and final sync of generated results to the writable `RD:` location.

################ CELL00B: ENVIRONMENT AND DEPENDENCIES ################
## Environment: `transcriber-env`

This notebook requires the **`transcriber-env`** conda environment.

---

## Required packages

| Package | Why it is needed |
|---|---|
| **Python 3.11** | Tested runtime for this notebook |
| **pandas** | Tables, summaries, and diarization export |
| **requests** + **urllib3** | HTTP client for WILLMA API calls |
| **soundfile** | Audio metadata and WAV read/write support |
| **scipy** | Filtering and resampling |
| **noisereduce** *(optional)* | Optional noise reduction |
| **openpyxl** | Writes diarization output as `.xlsx` |
| **reportlab** | Writes styled transcript PDFs |
| **ipykernel** | Registers the conda environment as a Jupyter kernel |
| **IPython** | `display()` for notebook tables |

### External tools

| Tool | Why it is needed |
|---|---|
| **ffmpeg** (on system PATH) | Extracts audio from container files before transcription |
| **rclone** (on system PATH) | Reads the public Research Drive share and writes outputs to the writable `RD:` location |

---
## `.whisper-env` requirements

Place a file named `.whisper-env` in the same folder as this notebook.

Required content:

```text
WILLMA_API_KEY=<YOUR_API_KEY>
WILLMA_BASE_URL=https://willma.surf.nl/api/v0
SHARE_LINK=https://hr.data.surf.nl/s/<share_token>
SHARE_PASSWORD=<share_password>
RD_OUTPUT_PATH=RD:HR-DATALAB-HEALTHCARE (Projectfolder)/TRANSCRIBED_WILLMA_OUTPUTS/
```

Optional keys:

```text
DEFAULT_INPUT_FOLDER=...  # ignored as input source in V05f
```

V05f uses this combined Research Drive pattern:
- `SHARE_LINK` + `SHARE_PASSWORD` are used only for reading the public audio share
- `RD_OUTPUT_PATH` must be a writable `RD:` destination such as `RD:HR-DATALAB-HEALTHCARE (Projectfolder)/TRANSCRIBED_WILLMA_OUTPUTS/`
- all generated transcript files are first written locally, then copied to `RD_OUTPUT_PATH`

If the env file is missing, `WILLMA_API_KEY` is empty, `SHARE_LINK` is missing, `SHARE_PASSWORD` is missing, or `RD_OUTPUT_PATH` is missing,
the setup cell will stop with a clear error.

In [31]:
################ CELL01: SETUP — PATHS, IMPORTS AND CONFIGURATION ################
from pathlib import Path
from urllib.parse import urlparse

NOTEBOOK_PATH = Path(
    r"D:\OneDrive - Hogeschool Rotterdam\SURF_PILOT\AI_HUB_PILOT\PYTHON\WILLMA_WHISPER_TRANSCRIBER_V05f.ipynb"
)
ENV_FILE = NOTEBOOK_PATH.with_name(".whisper-env")

import io
import json
import mimetypes
import shutil
import subprocess
import time

import pandas as pd
import requests
import soundfile as sf
from IPython.display import display
from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import (
    HRFlowable,
    KeepTogether,
    Paragraph,
    SimpleDocTemplate,
    Spacer,
    Table,
    TableStyle,
)
from requests.adapters import HTTPAdapter
from scipy import signal
from urllib3.util.retry import Retry

try:
    import noisereduce as nr  # type: ignore[reportMissingImports]
except ImportError:
    nr = None


def load_whisper_env(env_path):
    path = Path(env_path)
    if not path.exists():
        raise FileNotFoundError(
            "Missing env file. Create `.whisper-env` next to the notebook "
            "and define WILLMA_API_KEY, SHARE_LINK, SHARE_PASSWORD, and RD_OUTPUT_PATH."
        )

    env = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        env[key.strip()] = value.strip().strip('"').strip("'")
    return env


WHISPER_ENV = load_whisper_env(ENV_FILE)
SHARE_LINK = WHISPER_ENV.get("SHARE_LINK", "").strip()
SHARE_PASSWORD = WHISPER_ENV.get("SHARE_PASSWORD", "").strip()
RD_OUTPUT_PATH = WHISPER_ENV.get("RD_OUTPUT_PATH", "").strip()

if not SHARE_LINK:
    raise ValueError(
        "SHARE_LINK is not configured in `.whisper-env`. "
        "Add the Research Drive share URL before running this notebook."
    )

if not SHARE_PASSWORD:
    raise ValueError(
        "SHARE_PASSWORD is not configured in `.whisper-env`. "
        "Add the share password before running this notebook."
    )

if not RD_OUTPUT_PATH:
    raise ValueError(
        "RD_OUTPUT_PATH is not configured in `.whisper-env`. "
        "Add a writable `RD:` destination before running this notebook."
    )

DEFAULT_CACHE_ROOT = NOTEBOOK_PATH.with_name("research_drive_share_cache")
DEFAULT_OUTPUT_FOLDER = DEFAULT_CACHE_ROOT / "transcriber_v05f_outputs"
DEFAULT_CLEANED_FOLDER = DEFAULT_OUTPUT_FOLDER / "cleaned_audio"
DEFAULT_LOG_FOLDER = DEFAULT_OUTPUT_FOLDER / "logs"
DEFAULT_PROGRESS_FILE = DEFAULT_LOG_FOLDER / "batch_progress.json"
DEFAULT_REFERENCE_FOLDER = DEFAULT_OUTPUT_FOLDER / "references"
DEFAULT_INPUT_FOLDER = DEFAULT_CACHE_ROOT / "input_files"

for _d in [
    DEFAULT_CACHE_ROOT,
    DEFAULT_INPUT_FOLDER,
    DEFAULT_OUTPUT_FOLDER,
    DEFAULT_CLEANED_FOLDER,
    DEFAULT_LOG_FOLDER,
    DEFAULT_REFERENCE_FOLDER,
]:
    _d.mkdir(parents=True, exist_ok=True)

WILLMA_API_KEY = WHISPER_ENV.get("WILLMA_API_KEY", "").strip()
if not WILLMA_API_KEY or WILLMA_API_KEY == "<YOUR_API_KEY>":
    raise ValueError(
        "WILLMA_API_KEY is not configured in `.whisper-env`. "
        "Add a valid key before running this notebook."
    )


def create_retry_session(max_retries=None):
    n = max_retries or (CONFIG["max_retries"] if "CONFIG" in globals() else 3)
    session = requests.Session()
    retry = Retry(
        total=n,
        connect=n,
        read=n,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET", "POST", "PROPFIND"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def get_rclone_executable():
    for candidate in ["rclone.exe", "rclone"]:
        resolved = shutil.which(candidate)
        if resolved:
            return resolved
    raise FileNotFoundError(
        "rclone was not found on PATH. Install rclone before running this notebook."
    )


def extract_share_token(share_link):
    parsed = urlparse(str(share_link).strip())
    token = Path(parsed.path.rstrip("/")).name.strip()
    if not token:
        raise ValueError(
            "Could not extract a Research Drive share token from SHARE_LINK. "
            "Use a URL like `https://hr.data.surf.nl/s/<share_token>`."
        )
    return token


def obscure_rclone_password(share_password, rclone_executable):
    completed = subprocess.run(
        [rclone_executable, "obscure", share_password],
        capture_output=True,
        text=True,
        check=False,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )
    if completed.returncode != 0:
        raise RuntimeError(
            "rclone failed to obscure SHARE_PASSWORD. "
            f"stderr: {completed.stderr.strip() or completed.stdout.strip()}"
        )
    obscured = completed.stdout.strip()
    if not obscured:
        raise RuntimeError("rclone obscure returned an empty password value.")
    return obscured


def create_share_context(share_link, share_password):
    # Public-share input context: read-only access through the validated DAV URL.
    rclone_executable = get_rclone_executable()
    share_token = extract_share_token(share_link)
    obscured_password = obscure_rclone_password(share_password, rclone_executable)
    dav_url = f"https://hr.data.surf.nl/public.php/dav/files/{share_token}"
    rclone_source = (
        f":webdav,url='{dav_url}',vendor='nextcloud',"
        f"user='{share_token}',pass='{obscured_password}':"
    )
    return {
        "share_token": share_token,
        "share_url": share_link,
        "dav_url": dav_url,
        "rclone_source": rclone_source,
        "share_password": share_password,
        "rclone_executable": rclone_executable,
    }


def create_output_context(rd_output_path):
    # Writable output context: this must be a normal configured Research Drive
    # remote path such as RD:PROJECTS/WILLMA_OUTPUTS/.
    rclone_executable = get_rclone_executable()
    if not rd_output_path.startswith("RD:"):
        raise ValueError(
            "RD_OUTPUT_PATH must start with `RD:` and point to a writable Research Drive destination."
        )
    return {
        "rclone_executable": rclone_executable,
        "rclone_destination": rd_output_path,
    }


SHARE_CONTEXT = create_share_context(SHARE_LINK, SHARE_PASSWORD)
OUTPUT_CONTEXT = create_output_context(RD_OUTPUT_PATH)

CONFIG = {
    "input_folder": DEFAULT_INPUT_FOLDER,
    "output_folder": DEFAULT_OUTPUT_FOLDER,
    "cleaned_folder": DEFAULT_CLEANED_FOLDER,
    "reference_folder": DEFAULT_REFERENCE_FOLDER,
    "progress_file": DEFAULT_PROGRESS_FILE,
    "env_file": ENV_FILE,
    "share_link": SHARE_LINK,
    "share_password": SHARE_PASSWORD,
    "share_context": SHARE_CONTEXT,
    "rd_output_path": RD_OUTPUT_PATH,
    "output_context": OUTPUT_CONTEXT,
    "share_cache_root": DEFAULT_CACHE_ROOT,
    "willma_base_url": (
        WHISPER_ENV.get(
            "WILLMA_BASE_URL",
            "https://willma.surf.nl/api/v0",
        ).strip()
        or "https://willma.surf.nl/api/v0"
    ),
    "willma_api_key": WILLMA_API_KEY,
    "preferred_whisper_names": ["whisper-large-v3", "whisper"],
    "selected_whisper_model_name": None,
    "diarization_model": "pyannote/speaker-diarization",
    "supported_extensions": {
        ".3g2", ".3gp", ".aac", ".aif", ".aiff", ".alac", ".amr", ".ape",
        ".au", ".avi", ".caf", ".dts", ".eac3", ".f4a", ".f4b", ".flac",
        ".flv", ".m1a", ".m2a", ".m4a", ".m4b", ".m4p", ".m4v", ".mka",
        ".mkv", ".mov", ".mp2", ".mp3", ".mp4", ".mpa", ".mpc", ".mpeg",
        ".mpg", ".oga", ".ogg", ".ogm", ".ogv", ".opus", ".ra", ".rm",
        ".spx", ".ts", ".wav", ".weba", ".webm", ".wma", ".wmv",
    },
    "language": "nl",
    "request_timeout": 1800,
    "max_retries": 3,
    "sleep_between_attempts": 6,
    "max_diarization_attempts": 2,
    "max_audio_mb": 24,
    "overwrite_outputs": False,
    "preprocess_audio": False,
    "prefer_cleaned_audio": True,
    "target_sample_rate": 16000,
    "highpass_hz": 80,
    "noise_reduction_strength": 0.6,
    "peak_normalization_target": 0.98,
    "use_api_diarization": True,
    "alternative_pause_threshold": 1.2,
    "min_overlap_seconds": 0.15,
}

CONTAINER_EXTENSIONS = {
    ".3g2", ".3gp", ".aac", ".aif", ".aiff", ".alac", ".amr", ".ape", ".au",
    ".avi", ".caf", ".dts", ".eac3", ".f4a", ".f4b", ".flac", ".flv", ".m1a",
    ".m2a", ".m4a", ".m4b", ".m4p", ".m4v", ".mka", ".mkv", ".mov", ".mp2",
    ".mp3", ".mp4", ".mpa", ".mpc", ".mpeg", ".mpg", ".oga", ".ogg", ".ogm",
    ".ogv", ".opus", ".ra", ".rm", ".spx", ".ts", ".weba", ".webm", ".wma",
    ".wmv",
}
DIRECT_AUDIO_EXTENSIONS = {".wav"}

print(f"Using env file: {ENV_FILE}")
print(f"Research Drive share URL: {CONFIG['share_link']}")
print(f"Research Drive share token: {CONFIG['share_context']['share_token']}")
print(f"Research Drive DAV endpoint: {CONFIG['share_context']['dav_url']}")
print(f"Writable RD output path: {CONFIG['rd_output_path']}")
print(f"Local input cache: {CONFIG['input_folder']}")
print(f"Local output folder: {CONFIG['output_folder']}")
print(f"WILLMA base URL: {CONFIG['willma_base_url']}")
print("WILLMA API key loaded from env file.")

Using env file: D:\OneDrive - Hogeschool Rotterdam\SURF_PILOT\AI_HUB_PILOT\PYTHON\.whisper-env
Research Drive share URL: https://hr.data.surf.nl/s/BwGxSbX6WQmWdiS
Research Drive share token: BwGxSbX6WQmWdiS
Research Drive DAV endpoint: https://hr.data.surf.nl/public.php/dav/files/BwGxSbX6WQmWdiS
Writable RD output path: RD:HR-DATALAB-HEALTHCARE (Projectfolder)/TRANSCRIBED_WILLMA_OUTPUTS/
Local input cache: D:\OneDrive - Hogeschool Rotterdam\SURF_PILOT\AI_HUB_PILOT\PYTHON\research_drive_share_cache\input_files
Local output folder: D:\OneDrive - Hogeschool Rotterdam\SURF_PILOT\AI_HUB_PILOT\PYTHON\research_drive_share_cache\transcriber_v05f_outputs
WILLMA base URL: https://willma.surf.nl/api/v0
WILLMA API key loaded from env file.


In [ ]:
################ CELL03: ALL FUNCTIONS ################

# ── Path and shared-source helpers ───────────────────────────────────────────────
def normalize_path(p):
    return Path(p).expanduser().resolve()


def log_message(message, level="INFO"):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [{level}] {message}")
    try:
        import sys
        sys.stdout.flush()
    except Exception:
        pass


def clear_directory_contents(folder_path):
    folder = normalize_path(folder_path)
    if not folder.exists():
        return
    for path in sorted(folder.rglob("*"), reverse=True):
        try:
            if path.is_file() or path.is_symlink():
                path.unlink(missing_ok=True)
            elif path.is_dir():
                path.rmdir()
        except PermissionError:
            log_message(f"Could not remove cached path immediately: {path}", level="WARNING")
    folder.mkdir(parents=True, exist_ok=True)


def build_rclone_include_args(supported_extensions):
    """Build case-insensitive include rules so uppercase share files are copied too."""
    include_rules = []
    for ext in sorted(supported_extensions):
        lower_ext = ext.lower()
        upper_ext = ext.upper()
        include_rules.append(f"--include=*{lower_ext}")
        if upper_ext != lower_ext:
            include_rules.append(f"--include=*{upper_ext}")
    return include_rules


def sync_shared_audio_to_cache(share_link, share_password, local_input_folder):
    share_context = create_share_context(share_link, share_password)
    local_folder = normalize_path(local_input_folder)
    clear_directory_contents(local_folder)

    # Research Drive currently exposes files like `*.MP3` in uppercase. rclone's
    # include filters are case-sensitive, so we explicitly include both lower- and
    # upper-case variants for every supported extension.
    include_args = build_rclone_include_args(CONFIG["supported_extensions"])
    command = [
        share_context["rclone_executable"],
        "copy",
        share_context["rclone_source"],
        str(local_folder),
        "--progress",
        "--multi-thread-streams=4",
        "--transfers=4",
        "--checkers=4",
        "--links",
    ] + include_args

    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )
    if completed.returncode != 0:
        raise RuntimeError(
            "rclone failed to copy the Research Drive share files. "
            f"stderr: {completed.stderr.strip() or completed.stdout.strip()}"
        )

    downloaded_files = find_audio_files(local_folder)
    if not downloaded_files:
        log_message("No supported media files were copied from the configured Research Drive share.", level="WARNING")
    return downloaded_files


def sync_outputs_to_rd(local_output_folder, rd_output_path):
    # Output sync uses the writable RD: remote instead of the read-only public share.
    output_context = create_output_context(rd_output_path)
    local_folder = normalize_path(local_output_folder)
    if not local_folder.exists():
        raise FileNotFoundError(
            f"Local output folder does not exist: {local_folder}"
        )

    command = [
        output_context["rclone_executable"],
        "copy",
        str(local_folder),
        output_context["rclone_destination"],
        "--progress",
        "--create-empty-src-dirs",
    ]
    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )
    if completed.returncode != 0:
        raise RuntimeError(
            "rclone failed to copy local output files to RD_OUTPUT_PATH. "
            f"stderr: {completed.stderr.strip() or completed.stdout.strip()}"
        )
    return output_context["rclone_destination"]


def is_valid_media_file(path):
    p = normalize_path(path)
    if not p.exists() or not p.is_file() or p.stat().st_size <= 0:
        return False
    ext = p.suffix.lower()
    if ext in DIRECT_AUDIO_EXTENSIONS:
        try:
            sf.info(str(p))
            return True
        except Exception:
            return False
    if ext not in CONFIG["supported_extensions"]:
        return False
    probe_cmd = [
        "ffmpeg",
        "-v",
        "error",
        "-i",
        str(p),
        "-map",
        "0:a:0",
        "-f",
        "null",
        "-",
    ]
    try:
        completed = subprocess.run(probe_cmd, capture_output=True, text=True, check=False)
        return completed.returncode == 0
    except FileNotFoundError:
        return True


def find_audio_files(folder_path, supported_extensions=None):
    folder = normalize_path(folder_path)
    exts = supported_extensions or CONFIG["supported_extensions"]
    if not folder.exists():
        return []
    candidates = sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in exts
    )
    return [p for p in candidates if is_valid_media_file(p)]


def prepare_batch_file_list(folder_path, supported_extensions=None):
    return pd.DataFrame(
        [
            {
                "file_path": str(p),
                "file_name": p.name,
                "relative_path": str(p.relative_to(normalize_path(folder_path))),
                "extension": p.suffix.lower(),
            }
            for p in find_audio_files(folder_path, supported_extensions)
        ]
    )


# ── Audio validation ──────────────────────────────────────────────────────────────
def validate_audio_file(file_path, supported_extensions=None):
    path = normalize_path(file_path)
    exts = supported_extensions or CONFIG["supported_extensions"]
    result = {
        "file_path": str(path),
        "exists": path.exists(),
        "is_file": path.is_file(),
        "supported_extension": path.suffix.lower() in exts,
        "size_bytes": path.stat().st_size if path.exists() and path.is_file() else 0,
        "readable": False,
        "valid": False,
        "message": "",
    }
    if not result["exists"]:
        result["message"] = "File does not exist."
        return result
    if not result["is_file"]:
        result["message"] = "Path is not a file."
        return result
    if not result["supported_extension"]:
        result["message"] = "Unsupported media extension."
        return result
    if result["size_bytes"] <= 0:
        result["message"] = "File is empty."
        return result
    if path.suffix.lower() in DIRECT_AUDIO_EXTENSIONS:
        try:
            sf.info(str(path))
            result["readable"] = True
            result["valid"] = True
            result["message"] = "OK"
        except Exception as exc:
            result["message"] = f"Audio read failed: {exc}"
        return result

    result["readable"] = is_valid_media_file(path)
    result["valid"] = result["readable"]
    result["message"] = "OK (ffmpeg-readable media)" if result["valid"] else "Media is not readable by ffmpeg."
    return result


def run_preflight_checks(file_paths):
    return pd.DataFrame([validate_audio_file(p) for p in file_paths])


# ── Model catalog ─────────────────────────────────────────────────────────────────
def find_model_by_name_fragment(items, fragments):
    for frag in [(f or "").lower() for f in (fragments or [])]:
        for item in items:
            if frag in str(item.get("name", "")).lower():
                return item
    return None


def load_model_catalog(session=None):
    resp = (session or create_retry_session()).get(
        f"{CONFIG['willma_base_url']}/sequences",
        headers={"X-API-KEY": CONFIG["willma_api_key"], "Content-Type": "application/json"},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()


def load_whisper_model(session=None, preferred_names=None):
    catalog = load_model_catalog(session=session)
    sel = find_model_by_name_fragment(catalog, preferred_names or CONFIG["preferred_whisper_names"])
    if sel is None:
        sel = next((x for x in catalog if "whisper" in str(x.get("name", "")).lower()), None)
    if sel is None:
        raise ValueError("No Whisper model found in the WILLMA model catalog.")
    CONFIG["selected_whisper_model_name"] = sel.get("name")
    return sel


# ── Transcription helpers ─────────────────────────────────────────────────────────
def load_audio_metadata(file_path):
    path = normalize_path(file_path)
    if path.suffix.lower() in CONTAINER_EXTENSIONS:
        size_mb = path.stat().st_size / 1024 / 1024
        return {
            "file_path": str(path),
            "samplerate": None,
            "channels": None,
            "duration_seconds": None,
            "format": path.suffix.lstrip(".").upper(),
            "subtype": "container",
            "size_mb": round(size_mb, 2),
        }
    info = sf.info(str(path))
    return {
        "file_path": str(path),
        "samplerate": info.samplerate,
        "channels": info.channels,
        "duration_seconds": round(info.duration, 3),
        "format": info.format,
        "subtype": info.subtype,
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2),
    }


def extract_audio_from_container(src_path, out_dir=None):
    src = normalize_path(src_path)
    dst_dir = normalize_path(out_dir or CONFIG["cleaned_folder"])
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / f"{src.stem}.extracted.wav"

    if dst.exists() and not CONFIG["overwrite_outputs"]:
        log_message(f"{src.name}: extracted WAV already exists — reusing {dst.name}")
        return dst

    cmd = [
        "ffmpeg", "-y", "-i", str(src), "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", str(dst),
    ]

    try:
        log_message(f"{src.name}: extracting audio via ffmpeg → {dst.name} ({src.stat().st_size / 1024 / 1024:.1f} MB input)")
        create_no_window = getattr(subprocess, "CREATE_NO_WINDOW", 0)
        proc = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            creationflags=create_no_window,
        )
        if proc.returncode != 0:
            err = proc.stderr.decode(errors="replace")[-600:]
            log_message(f"{src.name}: ffmpeg failed (rc={proc.returncode}): {err}", level="WARNING")
            return None
        if not dst.exists() or dst.stat().st_size == 0:
            log_message(f"{src.name}: ffmpeg did not create a valid extracted WAV file", level="WARNING")
            return None
        out_mb = dst.stat().st_size / 1024 / 1024
        log_message(f"{src.name}: extraction done — {dst.name} ({out_mb:.1f} MB)")
        return dst
    except FileNotFoundError:
        log_message(f"{src.name}: ffmpeg not found on PATH — falling back to raw-bytes upload", level="WARNING")
        return None
    except Exception as exc:
        log_message(f"{src.name}: ffmpeg error: {exc}", level="WARNING")
        return None


def to_mono(audio):
    return audio.astype("float32") if getattr(audio, "ndim", 1) == 1 else audio.mean(axis=1).astype("float32")


def preprocess_audio(file_path, out_dir=None, config=None):
    cfg = config or CONFIG
    src = normalize_path(file_path)
    if src.suffix.lower() in CONTAINER_EXTENSIONS or src.stem.endswith(".extracted"):
        log_message(f"{src.name}: skipping DSP preprocessing (container or extracted WAV)")
        return None
    out_root = normalize_path(out_dir or cfg["cleaned_folder"])
    out_root.mkdir(parents=True, exist_ok=True)
    dst = out_root / f"{src.stem}.cleaned.wav"
    if dst.exists() and not cfg.get("overwrite_outputs", False):
        log_message(f"{src.name}: cleaned file already exists — skipping preprocessing")
        return dst

    mb = src.stat().st_size / 1024 / 1024
    log_message(f"{src.name}: reading audio ({mb:.1f} MB)")
    audio, sr = sf.read(str(src), always_2d=False)
    audio = to_mono(audio)
    dur = len(audio) / sr
    log_message(f"{src.name}: {dur/60:.1f} min, {sr} Hz — starting preprocessing")

    target_sr = int(cfg.get("target_sample_rate", 16000))
    if sr != target_sr:
        log_message(f"{src.name}: resampling {sr} Hz → {target_sr} Hz (this may take a while)")
        audio = signal.resample(audio, int(round(len(audio) * target_sr / sr))).astype("float32")
        sr = target_sr

    log_message(f"{src.name}: applying high-pass filter")
    b, a = signal.butter(4, min(cfg["highpass_hz"] / (sr / 2), 0.99), btype="highpass")
    audio = signal.filtfilt(b, a, audio).astype("float32")

    if nr is not None:
        log_message(f"{src.name}: reducing noise (this can take several minutes for long files)")
        noise = audio[: min(len(audio), int(sr * 1.5))]
        audio = nr.reduce_noise(
            y=audio,
            sr=sr,
            y_noise=noise,
            prop_decrease=cfg["noise_reduction_strength"],
            stationary=True,
        ).astype("float32")

In [33]:
################ CELL04: DIAGNOSTIC — SYNC AND DISCOVER FILES ################
# ── Diagnostic: sync and list all discovered audio files ─────────────────────────
downloaded_files = sync_shared_audio_to_cache(
    CONFIG["share_link"],
    CONFIG["share_password"],
    CONFIG["input_folder"],
)
print(f"Downloaded {len(downloaded_files)} shared file(s) into the local cache.")

found = find_audio_files(CONFIG["input_folder"])
print(f"Found {len(found)} supported audio file(s):")
for p in found:
    print(" ", p)

[2026-06-06 22:51:36] [WARNING] No supported media files were copied from the configured Research Drive share.
Downloaded 0 shared file(s) into the local cache.
Found 0 supported audio file(s):


In [34]:
################ CELL05: BATCH RUN / DEMO ################

# V05f: refresh the local cache from the public Research Drive share before each
# batch run so the notebook always processes the latest shared input files.
downloaded_files = sync_shared_audio_to_cache(
    CONFIG["share_link"],
    CONFIG["share_password"],
    CONFIG["input_folder"],
)
print(f"Shared cache refreshed with {len(downloaded_files)} file(s).")

session = create_retry_session()

if not str(CONFIG["willma_api_key"]).strip() or CONFIG["willma_api_key"] == "<YOUR_API_KEY>":
    raise ValueError(
        "CONFIG['willma_api_key'] is not set. Add WILLMA_API_KEY to `.whisper-env` before running the batch cell."
    )

try:
    selected_model = load_whisper_model(session=session)
    print(f"Selected model: {selected_model['name']}")
except requests.HTTPError as exc:
    status_code = getattr(exc.response, "status_code", None)
    if status_code == 401:
        raise ValueError(
            "WILLMA authentication failed (401 Unauthorized). Check whether WILLMA_API_KEY in `.whisper-env` contains a valid active API key."
        ) from exc
    raise

audio_file_df = prepare_batch_file_list(CONFIG["input_folder"])
display(audio_file_df)

if audio_file_df.empty or "file_path" not in audio_file_df.columns:
    display(pd.DataFrame([{
        "status": "no_audio_files_found",
        "input_folder": str(CONFIG["input_folder"]),
        "message": "No supported audio files were found in the shared Research Drive location, so no transcription or integrity check was run.",
    }]))
    batch_results = []
    verification_rows = []
else:
    valid_files = [str(p) for p in audio_file_df["file_path"].tolist()]
    preflight_df = run_preflight_checks(valid_files)
    display(preflight_df)

    print(f"Processing {len(valid_files)} file(s)...")
    forced_settings = dict(CONFIG)
    forced_settings["overwrite_outputs"] = True
    batch_results = []

    for fp in valid_files:
        try:
            batch_results.append(process_single_audio_file(fp, session=session, settings=forced_settings))
        except Exception as exc:
            batch_results.append({
                "file_path": fp,
                "status": "failed",
                "error": str(exc),
                "output_paths": {},
                "integrity": {"ok": False},
            })
            print(f"FAILED: {Path(fp).name} -> {exc}")

    display(summarize_batch_results(batch_results))

    verification_rows = []
    for item in batch_results:
        out = item.get("output_paths") or {}
        txt_path = Path(out.get("txt", "")) if out.get("txt") else None
        json_path = Path(out.get("json", "")) if out.get("json") else None
        xlsx_path = Path(out.get("xlsx", "")) if out.get("xlsx") else None
        srt_path = Path(out.get("srt", "")) if out.get("srt") else None
        pdf_path = Path(out.get("pdf", "")) if out.get("pdf") else None
        integrity_ok = bool((item.get("integrity") or {}).get("ok", False))
        verification_rows.append({
            "file": Path(item.get("file_path", "")).name,
            "status": item.get("status", ""),
            "error": item.get("error", ""),
            "txt": str(txt_path) if txt_path else "",
            "txt_exists": bool(txt_path and txt_path.exists()),
            "json_exists": bool(json_path and json_path.exists()),
            "xlsx_exists": bool(xlsx_path and xlsx_path.exists()),
            "srt_exists": bool(srt_path and srt_path.exists()),
            "pdf_exists": bool(pdf_path and pdf_path.exists()),
            "integrity_ok": integrity_ok,
            "txt_bytes": txt_path.stat().st_size if txt_path and txt_path.exists() else 0,
            "preview": txt_path.read_text(encoding="utf-8")[:120] if txt_path and txt_path.exists() else "",
        })

    rd_output_result = sync_outputs_to_rd(CONFIG["output_folder"], CONFIG["rd_output_path"])
    print(f"Local outputs copied to writable Research Drive destination: {rd_output_result}")

if verification_rows:
    display(pd.DataFrame(verification_rows))
else:
    display(pd.DataFrame([{
        "integrity_status": "not_run",
        "reason": "No transcription outputs were generated in this run.",
    }]))

[2026-06-06 22:51:37] [WARNING] No supported media files were copied from the configured Research Drive share.
Shared cache refreshed with 0 file(s).
Selected model: openai/whisper-large-v3


""


,status,input_folder,message
0,no_audio_files_found,D:\OneDrive - Hogeschool Rotterdam\SURF_PILOT\...,No supported audio files were found in the sha...


,integrity_status,reason
0,not_run,No transcription outputs were generated in thi...
